# Smart MCQ Solver — Ensemble (TF-IDF NN + DistilRoBERTa)

**Columns**: `id, prompt, A, B, C, D, E, answer`  
**Metric**: MAP@3  
**Submission**: `ID, Prediction` (space-separated top-3 letters)

**Architecture**:
```
TF-IDF NN (word+char)  → probs (CV ≈ 0.997)
DistilRoBERTa          → probs (prompt [SEP] option → correct/incorrect)
         ↓
  Weighted Ensemble → Rank → Top-3
```

> **Note**: DeBERTa v3 uses a custom CUDA kernel that crashes on Kaggle T4 GPUs.  
> DistilRoBERTa works on all GPUs and is faster to train.

In [2]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score
from difflib import SequenceMatcher
from scipy.sparse import hstack
from tqdm.auto import tqdm
import wandb

# Force CPU — Kaggle T4/P100 CUDA kernels are incompatible with this PyTorch build
device = torch.device('cpu')
print(f'Device: {device}')

Device: cpu


## 1. Load Data

In [3]:
train = pd.read_csv('/kaggle/input/datasets/kirandwippaul/mcqdata/train.csv')
test = pd.read_csv('/kaggle/input/datasets/kirandwippaul/mcqdata/test.csv')
print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Columns: {train.columns.tolist()}')

Train: (2000, 8) | Test: (500, 7)
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


## 2. Melt into (prompt, option) pairs

In [4]:
def melt(df, is_train=True):
    rows = []
    for _, row in df.iterrows():
        for opt in 'ABCDE':
            label = 1 if is_train and row['answer'] == opt else 0
            rows.append({'qid': row['id'], 'prompt': row['prompt'],
                         'opt': opt, 'opt_text': row[opt], 'label': label})
    return pd.DataFrame(rows)

train_melted = melt(train, True)
test_melted = melt(test, False)
print(f'Train melted: {train_melted.shape} | Test melted: {test_melted.shape}')

Train melted: (10000, 5) | Test melted: (2500, 5)


In [5]:
# Split by question id (no leakage)
np.random.seed(42)
uq = train['id'].unique().copy()
np.random.shuffle(uq)
split = int(0.8 * len(uq))
tr_ids, va_ids = set(uq[:split]), set(uq[split:])

tr = train_melted[train_melted['qid'].isin(tr_ids)].reset_index(drop=True)
va = train_melted[train_melted['qid'].isin(va_ids)].reset_index(drop=True)
print(f'Train pairs: {len(tr)} ({len(tr_ids)} Q) | Val pairs: {len(va)} ({len(va_ids)} Q)')

Train pairs: 8000 (1600 Q) | Val pairs: 2000 (400 Q)


## 3. Model A — TF-IDF Nearest Neighbour Lookup
Exploits near-duplicate structure between questions.

In [6]:
def make_text(row):
    return f"{row['prompt']} {row['A']} {row['B']} {row['C']} {row['D']} {row['E']}"

all_texts = pd.concat([train, test], ignore_index=True).apply(make_text, axis=1).tolist()
tfidf_w = TfidfVectorizer(ngram_range=(1,2), min_df=1, analyzer='word', sublinear_tf=True)
tfidf_c = TfidfVectorizer(ngram_range=(2,5), min_df=1, analyzer='char_wb', max_features=50000, sublinear_tf=True)
tfidf_w.fit(all_texts); tfidf_c.fit(all_texts)
print(f'Word feats: {len(tfidf_w.vocabulary_)} | Char feats: {len(tfidf_c.vocabulary_)}')

Word feats: 13798 | Char feats: 24842


In [7]:
train_txts = train.apply(make_text, axis=1).tolist()
test_txts = test.apply(make_text, axis=1).tolist()
train_vecs = hstack([tfidf_w.transform(train_txts), tfidf_c.transform(train_txts)])
test_vecs = hstack([tfidf_w.transform(test_txts), tfidf_c.transform(test_txts)])
print(f'Train vecs: {train_vecs.shape} | Test vecs: {test_vecs.shape}')

Train vecs: (2000, 38640) | Test vecs: (500, 38640)


In [8]:
K = 5

def tfidf_score(target_row, sims_row, ref_df):
    """Return {A: score, B: score, ...} using K nearest neighbours."""
    top_k = np.argsort(sims_row)[-K:][::-1]
    scores = {o: 0.0 for o in 'ABCDE'}
    for nn_i in top_k:
        nn = ref_df.iloc[nn_i]
        correct_text = str(nn[nn['answer']]).strip()
        w = sims_row[nn_i]
        for o in 'ABCDE':
            t = str(target_row[o]).strip()
            scores[o] += w * (1.0 if t == correct_text else SequenceMatcher(None, t, correct_text).ratio())
    return scores

In [9]:
# Get TF-IDF scores for every question in train (LOO) and test
train_sims = cosine_similarity(train_vecs, train_vecs)
np.fill_diagonal(train_sims, -1)  # exclude self-match

tfidf_train_scores = {}
for i in range(len(train)):
    tfidf_train_scores[train.iloc[i]['id']] = tfidf_score(train.iloc[i], train_sims[i], train)

test_sims = cosine_similarity(test_vecs, train_vecs)
tfidf_test_scores = {}
for i in range(len(test)):
    tfidf_test_scores[test.iloc[i]['id']] = tfidf_score(test.iloc[i], test_sims[i], train)

print(f'TF-IDF scored: {len(tfidf_train_scores)} train, {len(tfidf_test_scores)} test')

TF-IDF scored: 2000 train, 500 test


## 4. Model B — DistilRoBERTa
`prompt [SEP] option → correct/incorrect` binary classifier  
(DeBERTa v3 crashes on Kaggle T4 — `CUDA error: no kernel image available`)

In [10]:
class MCQDataset(Dataset):
    def __init__(self, df, tok, max_len=128, is_train=True):
        self.df, self.tok, self.max_len, self.is_train = df.reset_index(drop=True), tok, max_len, is_train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        enc = self.tok(r['prompt'], r['opt_text'], max_length=self.max_len,
                       padding='max_length', truncation=True, return_tensors='pt')
        out = {'input_ids': enc['input_ids'].squeeze(), 'attention_mask': enc['attention_mask'].squeeze()}
        if self.is_train: out['label'] = torch.tensor(r['label'], dtype=torch.long)
        return out

In [11]:
MODEL_NAME = 'distilroberta-base'

class TransformerClassifier(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(self.enc.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        h = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.head(self.drop(h))

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = TransformerClassifier(MODEL_NAME).to(device)
print(f'{MODEL_NAME} params: {sum(p.numel() for p in model.parameters()):,}')

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilroberta-base params: 82,119,938


In [12]:
MAX_LEN = 128
BS = 32
train_dl = DataLoader(MCQDataset(tr, tok, MAX_LEN), batch_size=BS, shuffle=True)
val_dl = DataLoader(MCQDataset(va, tok, MAX_LEN), batch_size=BS, shuffle=False)
optimizer = AdamW(model.parameters(), lr=2e-5)
sched = LinearLR(optimizer, start_factor=0.5, total_iters=2*len(train_dl))
crit = nn.CrossEntropyLoss()

wandb.init(project='smart-mcq-solver', config={
    'model': MODEL_NAME, 'max_len': MAX_LEN, 'bs': BS,
    'lr': 2e-5, 'epochs': 2, 'train_q': len(tr_ids), 'val_q': len(va_ids)
})
print(f'Train batches: {len(train_dl)} | Val batches: {len(val_dl)}')

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Train batches: 250 | Val batches: 63


In [13]:
def run_epoch(mdl, dl, optimizer=None, scheduler=None, epoch=0, phase='train'):
    is_train = optimizer is not None
    mdl.train() if is_train else mdl.eval()
    total_loss, preds, labels = 0, [], []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    pbar = tqdm(dl, desc=f'{phase} ep{epoch+1}', leave=False)
    with ctx:
        for step, b in enumerate(pbar):
            ids = b['input_ids'].to(device)
            mask = b['attention_mask'].to(device)
            lbl = b['label'].to(device)
            logits = mdl(ids, mask)
            loss = crit(logits, lbl)
            if is_train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
                optimizer.step(); scheduler.step()
                wandb.log({'batch/train_loss': loss.item(), 'batch/lr': scheduler.get_last_lr()[0]})
            total_loss += loss.item()
            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(lbl.cpu().numpy())
            pbar.set_postfix(loss=f'{loss.item():.4f}')
    acc = accuracy_score(labels, preds)
    return total_loss/len(dl), acc

In [14]:
best_acc = 0
for epoch in range(2):
    tr_loss, tr_acc = run_epoch(model, train_dl, optimizer, sched, epoch, 'train')
    va_loss, va_acc = run_epoch(model, val_dl, epoch=epoch, phase='val')
    
    wandb.log({
        'epoch': epoch+1,
        'train/loss': tr_loss, 'train/acc': tr_acc,
        'val/loss': va_loss, 'val/acc': va_acc,
    })
    print(f'Epoch {epoch+1}: Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | Val Loss={va_loss:.4f} Acc={va_acc:.4f}')
    
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save(model.state_dict(), '/tmp/roberta_best.pt')
        print(f'  ✓ Saved (best={best_acc:.4f})')

model.load_state_dict(torch.load('/tmp/roberta_best.pt', weights_only=True))
wandb.log({'best_val_acc': best_acc})
print(f'\nLoaded best {MODEL_NAME} (val acc={best_acc:.4f})')

train ep1:   0%|          | 0/250 [00:00<?, ?it/s]

val ep1:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 1: Train Loss=0.5043 Acc=0.7999 | Val Loss=0.4812 Acc=0.8000
  ✓ Saved (best=0.8000)


train ep2:   0%|          | 0/250 [00:00<?, ?it/s]

val ep2:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 2: Train Loss=0.3780 Acc=0.8420 | Val Loss=0.2449 Acc=0.9065
  ✓ Saved (best=0.9065)

Loaded best distilroberta-base (val acc=0.9065)


In [15]:
def get_transformer_scores(model, df, tok, device):
    """Return {qid: {A: prob, B: prob, ...}} from transformer."""
    model.eval()
    ds = MCQDataset(df, tok, MAX_LEN, is_train=False)
    dl = DataLoader(ds, batch_size=BS, shuffle=False)
    all_probs = []
    with torch.no_grad():
        for b in dl:
            logits = model(b['input_ids'].to(device), b['attention_mask'].to(device))
            all_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
    df = df.copy()
    df['prob'] = all_probs
    return {qid: dict(zip(g['opt'], g['prob'])) for qid, g in df.groupby('qid')}

In [16]:
rob_train = get_transformer_scores(model, train_melted, tok, device)
rob_test = get_transformer_scores(model, test_melted, tok, device)
print(f'DistilRoBERTa scored: {len(rob_train)} train, {len(rob_test)} test')

DistilRoBERTa scored: 2000 train, 500 test


## 5. Ensemble & Evaluate

In [17]:
def compute_map3(qids, pred_dict, answer_dict):
    maps = []
    for qid in qids:
        correct = answer_dict[qid]
        ranked = sorted(pred_dict[qid].items(), key=lambda x: x[1], reverse=True)
        top3 = [o for o, _ in ranked[:3]]
        ap = 1.0/(top3.index(correct)+1) if correct in top3 else 0.0
        maps.append(ap)
    return np.mean(maps)

answer_map = dict(zip(train['id'], train['answer']))
print('MAP@3 function ready')

MAP@3 function ready


In [18]:
def ensemble(tfidf_dict, rob_dict, w_tfidf, w_rob):
    """Weighted combination of two score dicts."""
    out = {}
    for qid in tfidf_dict:
        out[qid] = {}
        for o in 'ABCDE':
            t = tfidf_dict[qid].get(o, 0)
            d = rob_dict[qid].get(o, 0)
            out[qid][o] = w_tfidf * t + w_rob * d
    return out

## 6. Weight Search

In [20]:
answer_map = dict(zip(train['id'], train['answer']))
all_qids = list(answer_map.keys())

map_tfidf = compute_map3(all_qids, tfidf_train_scores, answer_map)
map_rob = compute_map3(all_qids, rob_train, answer_map)
print(f'{"TF-IDF alone":<30s} {map_tfidf:>10.6f}')
print(f'{"DistilRoBERTa alone":<30s} {map_rob:>10.6f}')
print('-'*55)

best_score, best_w = 0, 0
for w_tfidf in np.arange(0.3, 1.01, 0.05):
    w_rob = 1.0 - w_tfidf
    ens = ensemble(tfidf_train_scores, rob_train, w_tfidf, w_rob)
    score = compute_map3(all_qids, ens, answer_map)
    tag = ' <- best' if score > best_score else ''
    if score > best_score:
        best_score, best_w = score, w_tfidf
    print(f'TF-IDF={w_tfidf:.2f} RoBERTa={w_rob:.2f}       {score:>10.6f}{tag}')

print('='*55)
print(f'Best: TF-IDF={best_w:.2f} RoBERTa={1-best_w:.2f} -> MAP@3={best_score:.6f}')
wandb.log({'tfidf_map3': map_tfidf, 'roberta_map3': map_rob, 'ensemble_map3': best_score, 'best_tfidf_weight': best_w})

TF-IDF alone                     0.996917
DistilRoBERTa alone              0.953000
-------------------------------------------------------
TF-IDF=0.30 RoBERTa=0.70         0.995833 <- best
TF-IDF=0.35 RoBERTa=0.65         0.996167 <- best
TF-IDF=0.40 RoBERTa=0.60         0.996417 <- best
TF-IDF=0.45 RoBERTa=0.55         0.996667 <- best
TF-IDF=0.50 RoBERTa=0.50         0.996667
TF-IDF=0.55 RoBERTa=0.45         0.996667
TF-IDF=0.60 RoBERTa=0.40         0.996667
TF-IDF=0.65 RoBERTa=0.35         0.996667
TF-IDF=0.70 RoBERTa=0.30         0.996667
TF-IDF=0.75 RoBERTa=0.25         0.997167 <- best
TF-IDF=0.80 RoBERTa=0.20         0.997167
TF-IDF=0.85 RoBERTa=0.15         0.997167
TF-IDF=0.90 RoBERTa=0.10         0.997417 <- best
TF-IDF=0.95 RoBERTa=0.05         0.997417
TF-IDF=1.00 RoBERTa=0.00         0.996917
Best: TF-IDF=0.90 RoBERTa=0.10 -> MAP@3=0.997417


## 7. Generate Submission

In [23]:
# Use best weights

#final = ensemble(tfidf_test_scores, rob_test, best_w, 1-best_w)
final = ensemble(tfidf_test_scores, rob_test, 0.5, 0.5)


rows = []
for qid in sorted(final.keys()):
    ranked = sorted(final[qid].items(), key=lambda x: x[1], reverse=True)
    top3 = ' '.join(o for o, _ in ranked[:3])
    rows.append({'ID': qid, 'Prediction': top3})

submission = pd.DataFrame(rows)
submission.to_csv('submission.csv', index=False)
print(f'✓ Saved {len(submission)} predictions')
print(submission.head(10))

✓ Saved 500 predictions
   ID Prediction
0   1      A D B
1   2      B A C
2   3      B E D
3   4      E A D
4   5      C A D
5   6      D E A
6   7      E D C
7   8      B E A
8   9      C D A
9  10      B E C


In [22]:
#
wandb.finish()
print('\n wandb run finished')

batch/lr,▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇████
batch/train_loss,▆▅▄▄▆▆▄▄█▂▄▅▆▃▄▄▅▃▄▃▆▃▅▃▆▄▃▅▃▄▁▃▃▂▅▃▁▃▃▂
best_tfidf_weight,▁
best_val_acc,▁
ensemble_map3,▁
epoch,▁█
roberta_map3,▁
tfidf_map3,▁
train/acc,▁█
train/loss,█▁
+2,...



 wandb run finished
